# Census OCR - Gemini 3.1 Pro (API, no GPU)

End-to-end demo of the census OCR pipeline using Google's Gemini vision API.
Runs anywhere (local or Colab) with only an API key - no GPU, no model download.

1. `cp .env.example .env` and set `GEMINI_API_KEY` (get one at https://aistudio.google.com/apikey).
2. `pip install -r requirements.txt`
3. Run the cells below.


## 1. Setup: paths + imports


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from extract import extract_from_image, process_sheet, DEFAULT_MODEL
from compare import compare, print_report
print('Using model:', DEFAULT_MODEL)


## 2. Configure the run

1950 Bastrop ED 11-1, physical page 1 (`sheet_01.jpg`) vs ground-truth sheet `Bastrop 11-1`.


In [ ]:
CENSUS_YEAR = 1950
GROUND_TRUTH_SHEET = 'Bastrop 11-1'
PHYSICAL_PAGE = 1
IMAGE = PROJECT_ROOT / 'data/raw_images/1950_11-1/sheet_01.jpg'
GT = PROJECT_ROOT / 'data/ground_truth/Bastrop County 1950 Clean.xlsx'
OUT = PROJECT_ROOT / 'data/outputs/1950_11-1/sheet_01_extracted.json'
OUT.parent.mkdir(parents=True, exist_ok=True)
print('Image exists:', IMAGE.exists())
print('Ground truth exists:', GT.exists())


## 3. Extract (full page + overlapping crops + targeted retry)


In [ ]:
records = process_sheet(str(IMAGE), CENSUS_YEAR, str(OUT))
print('Extracted', len(records), 'records')
records[:3]


## 4. Compare against ground truth


In [ ]:
metrics, results = compare(str(OUT), str(GT), GROUND_TRUTH_SHEET, CENSUS_YEAR, PHYSICAL_PAGE)
print_report(metrics)


## 5. Inspect mismatches

Distinguish real extraction errors from known ground-truth limitations (see CLAUDE.md).


In [ ]:
for row in results:
    if not row['all_match']:
        bad = {f: v for f, v in row['fields'].items() if not v['match']}
        print('Line', row['line_number'], '->', {f: (v['extracted'], v['ground_truth']) for f, v in bad.items()})


## 6. Batch all 11 pages (optional)

Run from a shell: `python scripts/run_11_1_batch.py`. Or inline below.


In [ ]:
import subprocess
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts/run_11_1_batch.py')], cwd=str(PROJECT_ROOT))
